In [1]:
# ============================================================================
# CRITICAL FIX FOR WINDOWS UTF-8 ENCODING ISSUE
# This MUST be the first code cell and kernel MUST be restarted after adding this
# ============================================================================

import sys
import os

# Patch built-in open() to default to UTF-8
import builtins
_original_open = builtins.open

def utf8_open(file, mode='r', buffering=-1, encoding=None, errors=None, 
              newline=None, closefd=True, opener=None):
    if encoding is None and 'b' not in str(mode):
        encoding = 'utf-8'
    return _original_open(file, mode, buffering, encoding, errors, newline, closefd, opener)

builtins.open = utf8_open

# Set environment variables (helps for subprocesses)
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

print("✓ UTF-8 encoding patch applied")
print(f"  System: {sys.platform}")

# NOW import other libraries
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import evaluate
import pandas as pd
from tqdm import tqdm
import json
from pathlib import Path

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

✓ UTF-8 encoding patch applied
  System: win32


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please upd


Using device: cuda
PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3050
BF16 supported: True


## Configuration

Set up paths, language codes, and models to evaluate.

In [2]:
# Base model and directories
BASE_MODEL = "facebook/nllb-200-distilled-600M"
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")
DATA_DIR = Path("../data/splits")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# NLLB-200 language codes
NLLB_LANG_CODES = {
    "en": "eng_Latn",
    "tl": "tgl_Latn",
    "war": "war_Latn",
    "ceb": "ceb_Latn",
}

# Define models to evaluate
MODELS_TO_EVALUATE = {
    "cebuano": {
        "baseline": MODELS_DIR / "cebuano_baseline_nllb_lora_bf16" / "final_model",
        "experimental_stage1": MODELS_DIR / "cebuano_experimental_stage1_nllb_lora_bf16" / "final_model",
        "experimental_stage2": MODELS_DIR / "cebuano_experimental_stage2_nllb_lora_bf16" / "final_model",
        "bible_pair": "en-ceb",
        "src_lang": "eng_Latn",
        "tgt_lang": "ceb_Latn",
        "src_file": "en",
        "tgt_file": "ceb",
    },
    # Add more language pairs as needed
    "waray": {
        "baseline": MODELS_DIR / "waray_baseline_nllb_lora_bf16" / "final_model",
        "experimental_stage1": MODELS_DIR / "waray_experimental_stage1_nllb_lora_bf16" / "final_model",
        "experimental_stage2": MODELS_DIR / "waray_experimental_stage2_nllb_lora_bf16" / "final_model",
        "bible_pair": "en-war",
        "src_lang": "eng_Latn",
        "tgt_lang": "war_Latn",
        "src_file": "en",
        "tgt_file": "war",
    },
}

print("Configuration:")
print(f"  Base Model: {BASE_MODEL}")
print(f"  Models Directory: {MODELS_DIR}")
print(f"  Data Directory: {DATA_DIR}")
print(f"  Results Directory: {RESULTS_DIR}")
print(f"\nTarget Languages:")
for lang, config in MODELS_TO_EVALUATE.items():
    print(f"  - {lang.capitalize()}: {config['src_lang']} → {config['tgt_lang']} ({config['bible_pair']})")
    print(f"    Baseline: {config['baseline'].exists()}")
    print(f"    Stage 1:  {config['experimental_stage1'].exists()}")
    print(f"    Stage 2:  {config['experimental_stage2'].exists()}")

Configuration:
  Base Model: facebook/nllb-200-distilled-600M
  Models Directory: ..\models
  Data Directory: ..\data\splits
  Results Directory: ..\results

Target Languages:
  - Cebuano: eng_Latn → ceb_Latn (en-ceb)
    Baseline: True
    Stage 1:  True
    Stage 2:  True
  - Waray: eng_Latn → war_Latn (en-war)
    Baseline: True
    Stage 1:  True
    Stage 2:  True


## Load Bible Dataset Test Split

Load the test split from the Bible dataset for evaluation.

In [3]:
# Load split metadata
print("Loading Bible dataset test split...")
metadata_file = DATA_DIR / "split_metadata.json"

with open(metadata_file, "r", encoding="utf-8") as f:
    split_metadata = json.load(f)

print(f"\n Bible dataset metadata loaded!")
print(f"  Total sentences: {split_metadata['total_sentences']}")
print(f"  Test split size: {split_metadata['split_sizes']['test']}")
print(f"  Available pairs: {[pair['pair'] for pair in split_metadata['language_pairs']]}")

# Load test data for each language pair
bible_test_data = {}

for target_lang, config in MODELS_TO_EVALUATE.items():
    pair_name = config['bible_pair']
    pair_dir = DATA_DIR / pair_name
    
    try:
        print(f"\n{'='*80}")
        print(f"Loading: {target_lang.capitalize()} ({pair_name})")
        print(f"  Directory: {pair_dir}")
        print("="*80)
        
        # Load source language test file
        src_file = pair_dir / f"test.{config['src_file']}"
        print(f"  Loading {src_file.name}...")
        with open(src_file, "r", encoding="utf-8") as f:
            src_sentences = [line.strip() for line in f.readlines()]
        
        # Load target language test file
        tgt_file = pair_dir / f"test.{config['tgt_file']}"
        print(f"  Loading {tgt_file.name}...")
        with open(tgt_file, "r", encoding="utf-8") as f:
            tgt_sentences = [line.strip() for line in f.readlines()]
        
        # Verify same number of sentences
        if len(src_sentences) != len(tgt_sentences):
            print(f"  Warning: Mismatched lengths ({len(src_sentences)} vs {len(tgt_sentences)})")
        
        # Store data
        bible_test_data[target_lang] = {
            "source": src_sentences,
            "reference": tgt_sentences,
            "pair_name": pair_name,
        }
        print(f"   Loaded {len(src_sentences)} sentence pairs")
        
    except Exception as e:
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f" Bible test data loaded!")
print(f"  Language pairs: {list(bible_test_data.keys())}")
print("="*80)

Loading Bible dataset test split...

 Bible dataset metadata loaded!
  Total sentences: 4543
  Test split size: 455
  Available pairs: ['en-war', 'en-ceb', 'war-en', 'ceb-en']

Loading: Cebuano (en-ceb)
  Directory: ..\data\splits\en-ceb
  Loading test.en...
  Loading test.ceb...
   Loaded 455 sentence pairs

Loading: Waray (en-war)
  Directory: ..\data\splits\en-war
  Loading test.en...
  Loading test.war...
   Loaded 455 sentence pairs

 Bible test data loaded!
  Language pairs: ['cebuano', 'waray']


## Load Evaluation Metrics

Load BLEU and METEOR metrics for evaluation.

In [4]:
# Load BLEU and METEOR metrics
bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
print("✓ BLEU metric (sacrebleu) loaded successfully")
print("✓ METEOR metric loaded successfully")

✓ BLEU metric (sacrebleu) loaded successfully
✓ METEOR metric loaded successfully


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Translation Function

Function to translate sentences using NLLB-200 with LoRA adapters.

In [5]:
def translate_batch(model, tokenizer, sentences, src_lang, tgt_lang, batch_size=4, max_length=128):
    """Translate a batch of sentences using NLLB-200 with LoRA.
    
    Args:
        model: NLLB model with LoRA adapters or base model
        tokenizer: NLLB tokenizer
        sentences: List of source sentences
        src_lang: Source language code (e.g., 'eng_Latn')
        tgt_lang: Target language code (e.g., 'war_Latn')
        batch_size: Batch size for translation
        max_length: Maximum sequence length
    
    Returns:
        List of translated sentences
    """
    translations = []
    
    # Set source language for tokenizer
    tokenizer.src_lang = src_lang
    
    # Process in batches
    for i in tqdm(range(0, len(sentences), batch_size), desc="Translating"):
        batch = sentences[i:i+batch_size]
        
        # Tokenize source sentences
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Generate translations
        # Get forced_bos_token_id for target language
        if hasattr(tokenizer, 'lang_code_to_id'):
            forced_bos_token_id = tokenizer.lang_code_to_id.get(tgt_lang)
        else:
            forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
            if forced_bos_token_id == tokenizer.unk_token_id:
                forced_bos_token_id = None
        
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=max_length,
                num_beams=5,
                early_stopping=True
            )
        
        # Decode translations
        batch_translations = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )
        translations.extend(batch_translations)
    
    return translations

print("Translation function defined")

Translation function defined


## Evaluation Functions

Functions to evaluate models and compute BLEU and METEOR scores.

In [6]:
def evaluate_model(model_name, model_path, test_data, src_lang, tgt_lang, is_pretrained=False):
    """Evaluate a model on Bible test data.
    
    Args:
        model_name: Name of the model (for reporting)
        model_path: Path to model with LoRA adapters (or None for pretrained)
        test_data: Dictionary with 'source' and 'reference' lists
        src_lang: Source language code
        tgt_lang: Target language code
        is_pretrained: If True, load base model without LoRA adapters
    
    Returns:
        Dictionary with evaluation results
    """
    print("\n" + "="*80)
    print(f"Evaluating: {model_name}")
    if is_pretrained:
        print(f"Model: {BASE_MODEL} (Pretrained - No Fine-tuning)")
    else:
        print(f"Model path: {model_path}")
    print(f"Language pair: {src_lang} → {tgt_lang}")
    print("="*80 + "\n")
    
    # Load tokenizer and model
    try:
        print("1. Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        
        if is_pretrained:
            print("2. Loading pretrained base model...")
            model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
        else:
            # Check if model exists
            if not model_path.exists():
                print(f"Model not found: {model_path}")
                return None
                
            print("2. Loading base model...")
            base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
            
            print("3. Loading LoRA adapters...")
            model = PeftModel.from_pretrained(base_model, str(model_path))
        
        print("4. Moving to device...")
        model.to(device)
        model.eval()
        
        print("Model loaded successfully\n")
    except Exception as e:
        print(f"Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # Get test sentences
    source_sentences = test_data["source"]
    reference_sentences = test_data["reference"]
    
    print(f"5. Translating {len(source_sentences)} sentences...")
    translations = translate_batch(
        model, tokenizer, source_sentences, src_lang, tgt_lang
    )
    
    # Calculate BLEU score
    print("\n6. Computing BLEU score...")
    references = [[ref] for ref in reference_sentences]
    
    bleu_results = bleu_metric.compute(
        predictions=translations,
        references=references
    )
    
    # Calculate METEOR score
    print("7. Computing METEOR score...")
    meteor_results = meteor_metric.compute(
        predictions=translations,
        references=reference_sentences
    )
    
    # Clean up memory
    del model
    if not is_pretrained:
        del base_model
    torch.cuda.empty_cache()
    
    # Prepare results
    results = {
        "model_name": model_name,
        "model_path": "pretrained" if is_pretrained else str(model_path),
        "src_lang": src_lang,
        "tgt_lang": tgt_lang,
        "bleu_score": bleu_results["score"],
        "bleu_precisions": bleu_results["precisions"],
        "brevity_penalty": bleu_results["bp"],
        "meteor_score": meteor_results["meteor"],
        "length_ratio": bleu_results["sys_len"] / bleu_results["ref_len"],
        "translation_length": bleu_results["sys_len"],
        "reference_length": bleu_results["ref_len"],
        "num_sentences": len(translations),
        "sample_translations": [
            {
                "source": source_sentences[i],
                "reference": reference_sentences[i],
                "translation": translations[i]
            } for i in range(min(5, len(translations)))
        ]
    }
    
    print(f"\nEvaluation complete!")
    print(f"  BLEU Score:   {bleu_results['score']:.2f}")
    print(f"  METEOR Score: {meteor_results['meteor']:.4f}")
    print(f"  Precisions:   {[f'{p:.2f}' for p in bleu_results['precisions']]}")
    print(f"  Brevity Penalty: {bleu_results['bp']:.4f}")
    
    return results

print("Evaluation function defined")

Evaluation function defined


## Run Evaluation for All Models

Evaluate pretrained model, baseline, and experimental models on the Bible test split.

In [7]:
# Run evaluation for all models
all_results = {}

print("\n" + "#"*80)
print("# BIBLE DATASET EVALUATION - ALL MODELS")
print("#"*80)

for target_lang, config in MODELS_TO_EVALUATE.items():
    print(f"\n{'='*80}")
    print(f"Target Language: {target_lang.upper()}")
    print(f"Language Pair: {config['src_lang']} → {config['tgt_lang']}")
    print(f"Bible Pair: {config['bible_pair']}")
    print("="*80)
    
    # Get test data for this language pair
    if target_lang not in bible_test_data:
        print(f"No test data available for {target_lang}")
        continue
    
    lang_test_data = bible_test_data[target_lang]
    lang_results = {}
    
    # 1. Evaluate pretrained base model (no fine-tuning)
    print(f"\n{'='*80}")
    print(f"PRETRAINED BASE MODEL (No Fine-tuning)")
    print("="*80)
    result = evaluate_model(
        model_name=f"{target_lang.capitalize()} - Pretrained NLLB-200",
        model_path=None,
        test_data=lang_test_data,
        src_lang=config['src_lang'],
        tgt_lang=config['tgt_lang'],
        is_pretrained=True
    )
    if result:
        lang_results['pretrained'] = result
    
    # 2. Evaluate baseline model
    print(f"\n{'='*80}")
    print(f"BASELINE MODEL (Direct Fine-tuning)")
    print("="*80)
    if config['baseline'].exists():
        result = evaluate_model(
            model_name=f"{target_lang.capitalize()} - Baseline",
            model_path=config['baseline'],
            test_data=lang_test_data,
            src_lang=config['src_lang'],
            tgt_lang=config['tgt_lang']
        )
        if result:
            lang_results['baseline'] = result
    else:
        print(f"\nBaseline model not found: {config['baseline']}")
    
    # 3. Evaluate experimental stage 1 model
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL STAGE 1 MODEL (Similar Language Transfer)")
    print("="*80)
    if config['experimental_stage1'].exists():
        result = evaluate_model(
            model_name=f"{target_lang.capitalize()} - Experimental Stage 1",
            model_path=config['experimental_stage1'],
            test_data=lang_test_data,
            src_lang=config['src_lang'],
            tgt_lang=config['tgt_lang']
        )
        if result:
            lang_results['experimental_stage1'] = result
    else:
        print(f"\nExperimental Stage 1 model not found: {config['experimental_stage1']}")
    
    # 4. Evaluate experimental stage 2 model
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL STAGE 2 MODEL (Sequential Fine-tuning)")
    print("="*80)
    if config['experimental_stage2'].exists():
        result = evaluate_model(
            model_name=f"{target_lang.capitalize()} - Experimental Stage 2",
            model_path=config['experimental_stage2'],
            test_data=lang_test_data,
            src_lang=config['src_lang'],
            tgt_lang=config['tgt_lang']
        )
        if result:
            lang_results['experimental_stage2'] = result
    else:
        print(f"\nExperimental Stage 2 model not found: {config['experimental_stage2']}")
    
    # Store results for this language
    if lang_results:
        all_results[target_lang] = lang_results

print("\n" + "#"*80)
print(f"# EVALUATION COMPLETE - {len(all_results)} language(s) evaluated")
print("#"*80)


################################################################################
# BIBLE DATASET EVALUATION - ALL MODELS
################################################################################

Target Language: CEBUANO
Language Pair: eng_Latn → ceb_Latn
Bible Pair: en-ceb

PRETRAINED BASE MODEL (No Fine-tuning)

Evaluating: Cebuano - Pretrained NLLB-200
Model: facebook/nllb-200-distilled-600M (Pretrained - No Fine-tuning)
Language pair: eng_Latn → ceb_Latn

1. Loading tokenizer...
2. Loading pretrained base model...
4. Moving to device...
Model loaded successfully

5. Translating 455 sentences...


Translating: 100%|██████████| 114/114 [01:44<00:00,  1.09it/s]



6. Computing BLEU score...
7. Computing METEOR score...

Evaluation complete!
  BLEU Score:   26.27
  METEOR Score: 0.5415
  Precisions:   ['60.20', '33.08', '19.99', '12.13']
  Brevity Penalty: 0.9967

BASELINE MODEL (Direct Fine-tuning)

Evaluating: Cebuano - Baseline
Model path: ..\models\cebuano_baseline_nllb_lora_bf16\final_model
Language pair: eng_Latn → ceb_Latn

1. Loading tokenizer...
2. Loading base model...
3. Loading LoRA adapters...
4. Moving to device...
Model loaded successfully

5. Translating 455 sentences...


Translating: 100%|██████████| 114/114 [01:57<00:00,  1.03s/it]



6. Computing BLEU score...
7. Computing METEOR score...

Evaluation complete!
  BLEU Score:   29.22
  METEOR Score: 0.5735
  Precisions:   ['62.81', '36.27', '22.80', '14.38']
  Brevity Penalty: 0.9938

EXPERIMENTAL STAGE 1 MODEL (Similar Language Transfer)

Evaluating: Cebuano - Experimental Stage 1
Model path: ..\models\cebuano_experimental_stage1_nllb_lora_bf16\final_model
Language pair: eng_Latn → ceb_Latn

1. Loading tokenizer...
2. Loading base model...
3. Loading LoRA adapters...
4. Moving to device...
Model loaded successfully

5. Translating 455 sentences...


...


6. Computing BLEU score...
7. Computing METEOR score...

Evaluation complete!
  BLEU Score:   27.90
  METEOR Score: 0.5264
  Precisions:   ['60.35', '34.94', '22.29', '14.87']
  Brevity Penalty: 0.9648

EXPERIMENTAL STAGE 1 MODEL (Similar Language Transfer)

Evaluating: Waray - Experimental Stage 1
Model path: ..\models\waray_experimental_stage1_nllb_lora_bf16\final_model
Language pair: eng_Latn → war_Latn

1. Loading tokenizer...
2. Loading base model...
3. Loading LoRA adapters...
4. Moving to device...
Model loaded successfully

5. Translating 455 sentences...


Translating: 100%|██████████| 114/114 [02:05<00:00,  1.10s/it]



6. Computing BLEU score...
7. Computing METEOR score...

Evaluation complete!
  BLEU Score:   21.26
  METEOR Score: 0.4619
  Precisions:   ['53.02', '26.66', '15.26', '9.48']
  Brevity Penalty: 1.0000

EXPERIMENTAL STAGE 2 MODEL (Sequential Fine-tuning)

Evaluating: Waray - Experimental Stage 2
Model path: ..\models\waray_experimental_stage2_nllb_lora_bf16\final_model
Language pair: eng_Latn → war_Latn

1. Loading tokenizer...
2. Loading base model...
3. Loading LoRA adapters...
4. Moving to device...
Model loaded successfully

5. Translating 455 sentences...


Translating: 100%|██████████| 114/114 [02:00<00:00,  1.06s/it]



6. Computing BLEU score...
7. Computing METEOR score...

Evaluation complete!
  BLEU Score:   28.20
  METEOR Score: 0.5379
  Precisions:   ['61.73', '35.68', '22.69', '14.89']
  Brevity Penalty: 0.9602

################################################################################
# EVALUATION COMPLETE - 2 language(s) evaluated
################################################################################


## Display Results Table

Show evaluation results in a comprehensive table comparing all models.

In [8]:
# Prepare results for display
results_list = []

for target_lang, lang_results in all_results.items():
    for stage, result in lang_results.items():
        model_label = {
            'pretrained': 'Pretrained (Zero-shot)',
            'baseline': 'Baseline (Direct)',
            'experimental_stage1': 'Experimental (Stage 1)',
            'experimental_stage2': 'Experimental (Stage 2)'
        }.get(stage, stage)
        
        results_list.append({
            "Target Language": target_lang.capitalize(),
            "Model": model_label,
            "BLEU Score": f"{result['bleu_score']:.2f}",
            "METEOR": f"{result['meteor_score']:.4f}",
            "BLEU-1": f"{result['bleu_precisions'][0]:.2f}",
            "BLEU-2": f"{result['bleu_precisions'][1]:.2f}",
            "BLEU-3": f"{result['bleu_precisions'][2]:.2f}",
            "BLEU-4": f"{result['bleu_precisions'][3]:.2f}",
            "BP": f"{result['brevity_penalty']:.4f}",
        })

# Create DataFrame
results_df = pd.DataFrame(results_list)

# Display table
print("\n" + "="*100)
print("BIBLE DATASET EVALUATION RESULTS (NLLB-200 + LoRA + BF16)")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)


BIBLE DATASET EVALUATION RESULTS (NLLB-200 + LoRA + BF16)
Target Language                  Model BLEU Score METEOR BLEU-1 BLEU-2 BLEU-3 BLEU-4     BP
        Cebuano Pretrained (Zero-shot)      26.27 0.5415  60.20  33.08  19.99  12.13 0.9967
        Cebuano      Baseline (Direct)      29.22 0.5735  62.81  36.27  22.80  14.38 0.9938
        Cebuano Experimental (Stage 1)      25.05 0.5299  61.87  33.90  20.23  12.24 0.9330
        Cebuano Experimental (Stage 2)      29.42 0.5744  62.52  36.48  23.03  14.56 0.9948
          Waray Pretrained (Zero-shot)      20.93 0.4567  52.05  25.80  15.08   9.48 1.0000
          Waray      Baseline (Direct)      27.90 0.5264  60.35  34.94  22.29  14.87 0.9648
          Waray Experimental (Stage 1)      21.26 0.4619  53.02  26.66  15.26   9.48 1.0000
          Waray Experimental (Stage 2)      28.20 0.5379  61.73  35.68  22.69  14.89 0.9602


## Improvement Analysis

Analyze improvements from pretrained → baseline → experimental models.

In [9]:
# Calculate improvements
print("\n" + "="*100)
print("IMPROVEMENT ANALYSIS")
print("="*100)

for target_lang, lang_results in all_results.items():
    print(f"\n{target_lang.upper()}:")
    
    # Get scores for each model
    pretrained_bleu = lang_results.get('pretrained', {}).get('bleu_score', 0)
    pretrained_meteor = lang_results.get('pretrained', {}).get('meteor_score', 0)
    baseline_bleu = lang_results.get('baseline', {}).get('bleu_score', 0)
    baseline_meteor = lang_results.get('baseline', {}).get('meteor_score', 0)
    stage1_bleu = lang_results.get('experimental_stage1', {}).get('bleu_score', 0)
    stage1_meteor = lang_results.get('experimental_stage1', {}).get('meteor_score', 0)
    stage2_bleu = lang_results.get('experimental_stage2', {}).get('bleu_score', 0)
    stage2_meteor = lang_results.get('experimental_stage2', {}).get('meteor_score', 0)
    
    # Display scores
    if 'pretrained' in lang_results:
        print(f"  Pretrained (Zero-shot):    {pretrained_bleu:.2f} BLEU, {pretrained_meteor:.4f} METEOR")
    
    if 'baseline' in lang_results:
        print(f"  Baseline (Direct):         {baseline_bleu:.2f} BLEU, {baseline_meteor:.4f} METEOR")
        if 'pretrained' in lang_results:
            improvement = baseline_bleu - pretrained_bleu
            improvement_pct = (improvement / pretrained_bleu * 100) if pretrained_bleu > 0 else 0
            meteor_imp = baseline_meteor - pretrained_meteor
            meteor_imp_pct = (meteor_imp / pretrained_meteor * 100) if pretrained_meteor > 0 else 0
            print(f"    vs Pretrained:           {improvement:+.2f} BLEU ({improvement_pct:+.2f}%), {meteor_imp:+.4f} METEOR ({meteor_imp_pct:+.2f}%)")
    
    if 'experimental_stage1' in lang_results:
        print(f"  Experimental (Stage 1):    {stage1_bleu:.2f} BLEU, {stage1_meteor:.4f} METEOR")
    
    if 'experimental_stage2' in lang_results:
        print(f"  Experimental (Stage 2):    {stage2_bleu:.2f} BLEU, {stage2_meteor:.4f} METEOR")
        if 'pretrained' in lang_results:
            improvement = stage2_bleu - pretrained_bleu
            improvement_pct = (improvement / pretrained_bleu * 100) if pretrained_bleu > 0 else 0
            meteor_imp = stage2_meteor - pretrained_meteor
            meteor_imp_pct = (meteor_imp / pretrained_meteor * 100) if pretrained_meteor > 0 else 0
            print(f"    vs Pretrained:           {improvement:+.2f} BLEU ({improvement_pct:+.2f}%), {meteor_imp:+.4f} METEOR ({meteor_imp_pct:+.2f}%)")
    
    # Compare baseline vs experimental stage 2
    if 'baseline' in lang_results and 'experimental_stage2' in lang_results:
        improvement = stage2_bleu - baseline_bleu
        improvement_pct = (improvement / baseline_bleu * 100) if baseline_bleu > 0 else 0
        meteor_imp = stage2_meteor - baseline_meteor
        meteor_imp_pct = (meteor_imp / baseline_meteor * 100) if baseline_meteor > 0 else 0
        
        print(f"\n  Sequential vs Direct:")
        print(f"    Improvement:             {improvement:+.2f} BLEU ({improvement_pct:+.2f}%), {meteor_imp:+.4f} METEOR ({meteor_imp_pct:+.2f}%)")
        
        if improvement > 0:
            print(f"    Sequential fine-tuning IMPROVED over direct fine-tuning")
        elif improvement < 0:
            print(f"    Direct fine-tuning OUTPERFORMED sequential fine-tuning")
        else:
            print(f"    = No significant difference")

print("\n" + "="*100)


IMPROVEMENT ANALYSIS

CEBUANO:
  Pretrained (Zero-shot):    26.27 BLEU, 0.5415 METEOR
  Baseline (Direct):         29.22 BLEU, 0.5735 METEOR
    vs Pretrained:           +2.95 BLEU (+11.21%), +0.0320 METEOR (+5.92%)
  Experimental (Stage 1):    25.05 BLEU, 0.5299 METEOR
  Experimental (Stage 2):    29.42 BLEU, 0.5744 METEOR
    vs Pretrained:           +3.15 BLEU (+11.99%), +0.0329 METEOR (+6.08%)

  Sequential vs Direct:
    Improvement:             +0.20 BLEU (+0.70%), +0.0009 METEOR (+0.16%)
    Sequential fine-tuning IMPROVED over direct fine-tuning

WARAY:
  Pretrained (Zero-shot):    20.93 BLEU, 0.4567 METEOR
  Baseline (Direct):         27.90 BLEU, 0.5264 METEOR
    vs Pretrained:           +6.97 BLEU (+33.28%), +0.0697 METEOR (+15.27%)
  Experimental (Stage 1):    21.26 BLEU, 0.4619 METEOR
  Experimental (Stage 2):    28.20 BLEU, 0.5379 METEOR
    vs Pretrained:           +7.27 BLEU (+34.74%), +0.0812 METEOR (+17.78%)

  Sequential vs Direct:
    Improvement:             +0.31

## Sample Translations

Display sample translations from all models for qualitative comparison.

In [10]:
# Display sample translations
for target_lang, lang_results in all_results.items():
    print("\n" + "="*100)
    print(f"SAMPLE TRANSLATIONS - {target_lang.upper()}")
    print("="*100)
    
    # Order models for display
    model_order = ['pretrained', 'baseline', 'experimental_stage1', 'experimental_stage2']
    
    for stage in model_order:
        if stage not in lang_results:
            continue
            
        result = lang_results[stage]
        model_label = {
            'pretrained': 'Pretrained (Zero-shot)',
            'baseline': 'Baseline (Direct Fine-tuning)',
            'experimental_stage1': 'Experimental Stage 1 (Similar Language Transfer)',
            'experimental_stage2': 'Experimental Stage 2 (Sequential Fine-tuning)'
        }.get(stage, stage)
        
        print(f"\n{'-'*100}")
        print(f"{model_label}")
        print(f"BLEU: {result['bleu_score']:.2f} | METEOR: {result['meteor_score']:.4f}")
        print(f"{'-'*100}\n")
        
        for i, sample in enumerate(result['sample_translations'][:3], 1):
            print(f"Example {i}:")
            print(f"  Source:      {sample['source']}")
            print(f"  Reference:   {sample['reference']}")
            print(f"  Translation: {sample['translation']}")
            print()
    
    print("="*100)


SAMPLE TRANSLATIONS - CEBUANO

----------------------------------------------------------------------------------------------------
Pretrained (Zero-shot)
BLEU: 26.27 | METEOR: 0.5415
----------------------------------------------------------------------------------------------------

Example 1:
  Source:      for you know very well that the day of the Lord will come like a thief in the night.
  Reference:   Kay kamo nasayod na pag-ayo nga ang adlaw sa Ginoo moabot ra unya sama sa kawatan sa kagabhion.
  Translation: kay kamo nahibalo gayod nga ang Adlaw sa Ginoo moabut sama sa kawatan sa kagabhion.

Example 2:
  Source:      So Jacob’s sons did as he had commanded them:
  Reference:   Ug gibuhat sa iyang mga anak nga lalaki alang kaniya ang iyang gisugo kanila.
  Translation: Busa gibuhat sa mga anak nga lalake ni Jacob sumala sa iyang gisugo kanila.

Example 3:
  Source:      The people grieved for Benjamin, because the Lord had made a gap in the tribes of Israel.
  Reference:   Ug 

## Save Results

Save evaluation results to JSON file.

In [11]:
# Save results to JSON
output_file = RESULTS_DIR / "bible_dataset_all_models_evaluation.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\n Results saved to: {output_file}")
print(f"  Total languages evaluated: {len(all_results)}")
for target_lang, lang_results in all_results.items():
    print(f"    {target_lang.capitalize()}: {len(lang_results)} model(s)")

# Also save as CSV for easy viewing
csv_data = []
for target_lang, lang_results in all_results.items():
    for stage, result in lang_results.items():
        csv_data.append({
            'Target Language': target_lang,
            'Model Stage': stage,
            'Model Name': result['model_name'],
            'BLEU Score': result['bleu_score'],
            'METEOR Score': result['meteor_score'],
            'BLEU-1': result['bleu_precisions'][0],
            'BLEU-2': result['bleu_precisions'][1],
            'BLEU-3': result['bleu_precisions'][2],
            'BLEU-4': result['bleu_precisions'][3],
            'Brevity Penalty': result['brevity_penalty'],
            'Translation Length': result['translation_length'],
            'Reference Length': result['reference_length'],
            'Length Ratio': result['length_ratio'],
        })

csv_df = pd.DataFrame(csv_data)
csv_file = RESULTS_DIR / "bible_dataset_all_models_evaluation.csv"
csv_df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"  CSV saved to: {csv_file}")


 Results saved to: ..\results\bible_dataset_all_models_evaluation.json
  Total languages evaluated: 2
    Cebuano: 4 model(s)
    Waray: 4 model(s)
  CSV saved to: ..\results\bible_dataset_all_models_evaluation.csv
